# Module 04: MLflow

## What is MLflow and why does it exist?

Without experiment tracking, ML development looks like this:
- You run K-Means with 120 clusters, silhouette score is 0.41
- You run it again with 80 clusters, score is 0.47 — better!
- You try UMAP with different neighbors and get confused about which combo worked
- Three weeks later you can't reproduce the best result because you don't remember the parameters

**MLflow solves this.** It logs every experiment run (parameters, metrics, output artifacts) so you can compare, reproduce, and promote models to production.

MLflow has four components:

| Component | What it does | Analogy |
|---|---|---|
| **Tracking** | Logs parameters, metrics, tags, artifacts for each run | Git commit for ML runs |
| **Projects** | Packages code for reproducibility (we won't use this much) | Docker for ML code |
| **Models** | Standard format for packaging trained models | Standardized container |
| **Registry** | Versioned model store with lifecycle stages | npm/PyPI for ML models |

## Key Vocabulary

- **Experiment** — a named group of related runs (e.g., `kmeans-clustering-experiments`)
- **Run** — one execution of your training code. Has a unique ID, timestamps, and logged data.
- **Parameter** — an input to your training (e.g., `n_clusters=120`). Logged once.
- **Metric** — a measured output (e.g., `silhouette_score=0.47`). Can be logged multiple times (e.g., loss per epoch).
- **Artifact** — any file output (model pickle, plot, CSV). Stored in the artifact store (local filesystem or S3).
- **Model Registry** — a central catalog of registered model versions with stages: `None` → `Staging` → `Production` → `Archived`


## Setup

```bash
pip install mlflow scikit-learn pandas numpy
```

MLflow can run without a server (logs to `./mlruns/` locally). For the UI, start the server in a separate terminal:

```bash
mlflow server --host 0.0.0.0 --port 5000
# Then open http://localhost:5000
```

Or just run the cells below — `mlflow.set_tracking_uri()` defaults to local file storage, and you can open the UI anytime.

In [ ]:
import mlflow
import mlflow.sklearn
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.datasets import make_blobs

# Use local file tracking (default — no server needed)
# To use a server: mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_tracking_uri("./mlruns")

print(f"MLflow version: {mlflow.__version__}")
print(f"Tracking URI: {mlflow.get_tracking_uri()}")

---
## Exercise 1: Log Your First Run

We'll generate synthetic embedding data (like creator embeddings) and run K-Means, logging everything to MLflow.

In [ ]:
# Generate synthetic data resembling creator embeddings
# 500 "creators", 64-dimensional "embeddings", 10 true clusters
X, true_labels = make_blobs(n_samples=500, n_features=64, centers=10, random_state=42)
print(f"Data shape: {X.shape}")

In [ ]:
# Set the experiment (creates it if it doesn't exist)
mlflow.set_experiment("kmeans-clustering")

# Run with explicit context manager
with mlflow.start_run(run_name="baseline_k10"):
    # --- Parameters (inputs to training) ---
    n_clusters = 10
    random_state = 42
    mlflow.log_param("n_clusters", n_clusters)
    mlflow.log_param("random_state", random_state)
    mlflow.log_param("data_size", len(X))

    # --- Training ---
    model = KMeans(n_clusters=n_clusters, random_state=random_state, n_init=10)
    labels = model.fit_predict(X)

    # --- Metrics (outputs of training) ---
    sil_score = silhouette_score(X, labels)
    db_score = davies_bouldin_score(X, labels)
    inertia = model.inertia_

    mlflow.log_metric("silhouette_score", sil_score)
    mlflow.log_metric("davies_bouldin_score", db_score)
    mlflow.log_metric("inertia", inertia)

    # --- Log the model ---
    mlflow.sklearn.log_model(model, artifact_path="kmeans_model")

    # Get the run ID for reference
    run_id = mlflow.active_run().info.run_id
    print(f"Run ID: {run_id}")
    print(f"Silhouette Score: {sil_score:.4f}")
    print(f"Davies-Bouldin Score: {db_score:.4f} (lower is better)")
    print(f"Inertia: {inertia:.2f}")

In [ ]:
# You can now open the MLflow UI to see this run:
# Terminal: mlflow ui --port 5000
# Browser: http://localhost:5000

# Or query it programmatically
client = mlflow.tracking.MlflowClient()
experiment = client.get_experiment_by_name("kmeans-clustering")
runs = client.search_runs(experiment.experiment_id)
print(f"Runs in experiment: {len(runs)}")
for run in runs:
    print(f"  {run.info.run_name}: silhouette={run.data.metrics.get('silhouette_score', 'N/A'):.4f}")

---
## Exercise 2: Compare Multiple Runs

This is the core MLflow workflow — run experiments with different hyperparameters and compare results. This is exactly what you'll do when tuning K-Means for the Galaxy project.

In [ ]:
mlflow.set_experiment("kmeans-clustering")

# Try different numbers of clusters
cluster_options = [5, 10, 20, 30, 50]

for n in cluster_options:
    with mlflow.start_run(run_name=f"k{n}"):
        mlflow.log_param("n_clusters", n)
        mlflow.log_param("n_init", 10)

        model = KMeans(n_clusters=n, random_state=42, n_init=10)
        labels = model.fit_predict(X)

        sil = silhouette_score(X, labels)
        db = davies_bouldin_score(X, labels)

        mlflow.log_metric("silhouette_score", sil)
        mlflow.log_metric("davies_bouldin_score", db)
        mlflow.log_metric("inertia", model.inertia_)

        print(f"n_clusters={n:3d}: silhouette={sil:.4f}, db={db:.4f}")

In [ ]:
# Find the best run by silhouette score
client = mlflow.tracking.MlflowClient()
experiment = client.get_experiment_by_name("kmeans-clustering")

best_runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.silhouette_score DESC"],
    max_results=3
)

print("Top 3 runs by silhouette score:")
for run in best_runs:
    params = run.data.params
    metrics = run.data.metrics
    print(f"  n_clusters={params['n_clusters']:>3} | "
          f"silhouette={metrics['silhouette_score']:.4f} | "
          f"run_id={run.info.run_id[:8]}...")

---
## Exercise 3: Log Artifacts

Artifacts are any file output — plots, CSVs, model binaries. MLflow stores them alongside the run metadata.

In [ ]:
import matplotlib.pyplot as plt
import os
import tempfile

mlflow.set_experiment("kmeans-clustering")

with mlflow.start_run(run_name="with_artifacts"):
    n_clusters = 10
    mlflow.log_param("n_clusters", n_clusters)

    model = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    labels = model.fit_predict(X)

    sil = silhouette_score(X, labels)
    mlflow.log_metric("silhouette_score", sil)

    with tempfile.TemporaryDirectory() as tmp:
        # --- Artifact 1: A plot ---
        fig, ax = plt.subplots(figsize=(8, 6))
        scatter = ax.scatter(X[:, 0], X[:, 1], c=labels, cmap='tab10', s=10, alpha=0.7)
        ax.set_title(f"K-Means Clusters (k={n_clusters}, silhouette={sil:.3f})")
        plt.colorbar(scatter, ax=ax, label='Cluster')
        plot_path = os.path.join(tmp, "cluster_plot.png")
        fig.savefig(plot_path, dpi=100, bbox_inches='tight')
        plt.close()
        mlflow.log_artifact(plot_path, artifact_path="plots")

        # --- Artifact 2: A summary CSV ---
        import pandas as pd
        cluster_sizes = pd.Series(labels).value_counts().sort_index()
        df = pd.DataFrame({'cluster_id': cluster_sizes.index, 'size': cluster_sizes.values})
        csv_path = os.path.join(tmp, "cluster_sizes.csv")
        df.to_csv(csv_path, index=False)
        mlflow.log_artifact(csv_path, artifact_path="data")

    # --- Log the model ---
    mlflow.sklearn.log_model(model, artifact_path="kmeans_model")

    print(f"Run complete. Check the MLflow UI to see the plot and CSV.")
    print(f"Run ID: {mlflow.active_run().info.run_id}")

---
## Exercise 4: Model Registry

The Model Registry is where you promote models from experimentation to production. In the Galaxy pipeline, Airflow checks the registry before deploying a new K-Means model — it only proceeds if the silhouette score is above a threshold AND the model is tagged as `Production`.

**Lifecycle stages:**
- `None` — just registered, not ready
- `Staging` — candidate for production, being evaluated
- `Production` — the live model serving real traffic
- `Archived` — retired

In [ ]:
# First, train a model with a run
mlflow.set_experiment("kmeans-clustering")

with mlflow.start_run(run_name="registry_demo") as run:
    mlflow.log_param("n_clusters", 10)
    model = KMeans(n_clusters=10, random_state=42, n_init=10)
    model.fit(X)
    sil = silhouette_score(X, model.labels_)
    mlflow.log_metric("silhouette_score", sil)

    # Register the model directly during the run
    # This creates the registered model if it doesn't exist, then adds version 1
    model_uri = f"runs:/{run.info.run_id}/kmeans_model"
    mlflow.sklearn.log_model(
        model,
        artifact_path="kmeans_model",
        registered_model_name="galaxy-kmeans"  # <-- registers it
    )

    run_id = run.info.run_id
    print(f"Run ID: {run_id}")
    print(f"Model URI: {model_uri}")

In [ ]:
# Work with the registry using MlflowClient
client = mlflow.tracking.MlflowClient()

# List all versions of the registered model
versions = client.search_model_versions("name='galaxy-kmeans'")
for v in versions:
    print(f"Version {v.version}: stage={v.current_stage}, run_id={v.run_id[:8]}...")

In [ ]:
# Transition to Production
# In the Galaxy pipeline, this happens automatically in Airflow
# after the silhouette score passes a threshold
latest_version = versions[0].version

client.transition_model_version_stage(
    name="galaxy-kmeans",
    version=latest_version,
    stage="Production"
)
print(f"Version {latest_version} promoted to Production")

In [ ]:
# Load the Production model — this is how the inference pipeline uses it
production_model = mlflow.sklearn.load_model("models:/galaxy-kmeans/Production")

# Predict cluster for a new "creator" (random 64-dim vector)
new_creator_embedding = np.random.randn(1, 64)
cluster = production_model.predict(new_creator_embedding)
print(f"New creator assigned to cluster: {cluster[0]}")
print(f"Model type: {type(production_model)}")

---
## Exercise 5: Auto-logging

MLflow can automatically log sklearn parameters and metrics without explicit `log_param`/`log_metric` calls. Useful for quick experiments.

In [ ]:
mlflow.set_experiment("kmeans-autolog-demo")

# Enable autologging for sklearn
mlflow.sklearn.autolog()

with mlflow.start_run(run_name="autologged"):
    model = KMeans(n_clusters=15, random_state=42, n_init=10)
    model.fit(X)
    # MLflow automatically logged: n_clusters, n_init, random_state, inertia, n_iter_
    # Add a custom metric manually
    mlflow.log_metric("silhouette_score", silhouette_score(X, model.labels_))

print("Check MLflow UI — all KMeans params were logged automatically")

# Turn autolog off when you want fine-grained control
mlflow.sklearn.autolog(disable=True)

---
## How MLflow Maps to the Galaxy Project

In the real pipeline:

```python
# Inside training/train.py (the SageMaker / k8s Job container)
import mlflow

# Tracking server runs in k3s — point to it
mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])  # e.g. http://mlflow-service:5000
mlflow.set_experiment("galaxy-kmeans-production")

with mlflow.start_run():
    mlflow.log_params({"n_clusters": 120, "umap_neighbors": 30, "model": "gte-large-en-v1.5"})
    
    # ... run embedding + clustering ...
    
    mlflow.log_metrics({"silhouette_score": sil, "davies_bouldin": db})
    mlflow.sklearn.log_model(kmeans, artifact_path="kmeans_model",
                             registered_model_name="galaxy-kmeans")
```

Then in Airflow (`training_pipeline_dag.py`):
```python
# After training completes, check if the model is good enough to promote
client = mlflow.tracking.MlflowClient(tracking_uri=MLFLOW_URI)
latest_run = client.search_runs(..., order_by=["start_time DESC"])[0]
sil = latest_run.data.metrics["silhouette_score"]

if sil >= 0.45:
    client.transition_model_version_stage("galaxy-kmeans", latest_version, "Production")
    # ... continue pipeline: upload to Oracle, reload Weaviate, etc.
else:
    raise AirflowException(f"Model quality insufficient: silhouette={sil:.3f} < 0.45")
```

This is the **quality gate pattern** — MLflow is the checkpoint between training and deployment.

## Summary

| Action | Code |
|---|---|
| Start a run | `with mlflow.start_run():` |
| Log a parameter | `mlflow.log_param("key", value)` |
| Log a metric | `mlflow.log_metric("name", value)` |
| Log a file | `mlflow.log_artifact("path/to/file")` |
| Log + register a model | `mlflow.sklearn.log_model(model, ..., registered_model_name="name")` |
| Promote a version | `client.transition_model_version_stage("name", version, "Production")` |
| Load production model | `mlflow.sklearn.load_model("models:/name/Production")` |